In [ ]:
#Self-Attention, MQA, GQA, FlashAttention v2, RoPE

# Self-Attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super(SelfAttention, self).__init__()
        self.embed_dim = embed_dim

        # Khởi tạo các ma trận trọng số Query, Key, Value
        self.W_Q = nn.Linear(embed_dim, embed_dim)
        self.W_K = nn.Linear(embed_dim, embed_dim)
        self.W_V = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x shape: (batch_size, seq_length, embed_dim)

        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # Tính Attention Scores
        attention_scores = torch.bmm(Q, K.transpose(1, 2)) / (self.embed_dim ** 0.5)

        # Tính Attention Weights
        attention_weights = F.softmax(attention_scores, dim=-1)

        # Attention Output
        attention_output = torch.bmm(attention_weights, V)

        return attention_output, attention_weights


# ===== Ví dụ minh họa =====

# Giả lập dữ liệu đầu vào đơn giản
# Giả sử 1 batch, độ dài chuỗi là 3, embedding dimension là 4
example_input = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                               [0.0, 2.0, 0.0, 2.0],
                               [1.0, 1.0, 1.0, 1.0]]])

print("Đầu vào:\n", example_input)

# Khởi tạo mô hình Self-Attention
embed_dim = 4
self_attention = SelfAttention(embed_dim)

# Forward
output, attn_weights = self_attention(example_input)

print("\nMa trận Attention Weights:\n", attn_weights)
print("\nKết quả đầu ra của Self-Attention:\n", output)

Đầu vào:
 tensor([[[1., 0., 1., 0.],
         [0., 2., 0., 2.],
         [1., 1., 1., 1.]]])

Ma trận Attention Weights:
 tensor([[[0.3238, 0.3138, 0.3624],
         [0.4046, 0.1633, 0.4321],
         [0.3607, 0.2211, 0.4182]]], grad_fn=<SoftmaxBackward0>)

Kết quả đầu ra của Self-Attention:
 tensor([[[-0.5269, -0.7646, -0.9176, -1.1321],
         [-0.4630, -0.8377, -0.9515, -0.9425],
         [-0.4909, -0.8135, -0.9414, -1.0250]]], grad_fn=<BmmBackward0>)


# MQA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiQueryAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(MultiQueryAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        # Một Query cho mỗi head
        self.W_Q = nn.Linear(embed_dim, embed_dim)
        # Key và Value chung cho tất cả heads
        self.W_K = nn.Linear(embed_dim, self.head_dim)
        self.W_V = nn.Linear(embed_dim, self.head_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        batch_size, seq_length, _ = x.size()

        Q = self.W_Q(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_K(x).unsqueeze(1)  # dùng chung cho tất cả heads
        V = self.W_V(x).unsqueeze(1)  # dùng chung cho tất cả heads

        # Tính Attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Tính Attention weights
        attn_weights = F.softmax(attn_scores, dim=-1)

        # Tính output
        attn_output = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(batch_size, seq_length, self.embed_dim)

        output = self.out_proj(attn_output)

        return output, attn_weights


# ====== Ví dụ minh họa ======

# Giả lập đầu vào: batch_size=1, seq_length=3, embed_dim=4
example_input = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                               [0.0, 2.0, 0.0, 2.0],
                               [1.0, 1.0, 1.0, 1.0]]])

print("Đầu vào:\n", example_input)

# Khởi tạo mô hình Multi-Query Attention
embed_dim = 4
num_heads = 2
mqa = MultiQueryAttention(embed_dim, num_heads)

# Forward
output, attn_weights = mqa(example_input)

print("\nMa trận Attention Weights:\n", attn_weights)
print("\nKết quả đầu ra của Multi-Query Attention:\n", output)


Đầu vào:
 tensor([[[1., 0., 1., 0.],
         [0., 2., 0., 2.],
         [1., 1., 1., 1.]]])

Ma trận Attention Weights:
 tensor([[[[0.2605, 0.4117, 0.3279],
          [0.1742, 0.5540, 0.2718],
          [0.1987, 0.4987, 0.3025]],

         [[0.2783, 0.3811, 0.3406],
          [0.5271, 0.1290, 0.3439],
          [0.4014, 0.2339, 0.3648]]]], grad_fn=<SoftmaxBackward0>)

Kết quả đầu ra của Multi-Query Attention:
 tensor([[[ 0.1184, -0.5489, -0.2269,  0.0653],
         [ 0.0795, -0.7978, -0.1937,  0.3328],
         [ 0.0905, -0.6939, -0.2111,  0.2183]]], grad_fn=<ViewBackward0>)


# GQA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GroupedQueryAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, num_key_value_groups):
        super(GroupedQueryAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_groups = num_key_value_groups
        self.head_dim = embed_dim // num_heads

        assert self.head_dim * num_heads == embed_dim, "embed_dim phải chia hết cho num_heads"
        assert num_heads % num_key_value_groups == 0, "num_heads phải chia hết cho num_key_value_groups"

        self.W_Q = nn.Linear(embed_dim, embed_dim)
        self.W_K = nn.Linear(embed_dim, self.num_groups * self.head_dim)
        self.W_V = nn.Linear(embed_dim, self.num_groups * self.head_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        batch_size, seq_length, _ = x.size()

        Q = self.W_Q(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)

        K = self.W_K(x).view(batch_size, seq_length, self.num_groups, self.head_dim).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_length, self.num_groups, self.head_dim).transpose(1, 2)

        K = K.unsqueeze(2).repeat(1, 1, self.num_heads // self.num_groups, 1, 1).view(batch_size, self.num_heads, seq_length, self.head_dim)
        V = V.unsqueeze(2).repeat(1, 1, self.num_heads // self.num_groups, 1, 1).view(batch_size, self.num_heads, seq_length, self.head_dim)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)

        attn_output = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(batch_size, seq_length, self.embed_dim)

        output = self.out_proj(attn_output)

        return output, attn_weights


# ====== Ví dụ minh họa ======

example_input = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                               [0.0, 2.0, 0.0, 2.0],
                               [1.0, 1.0, 1.0, 1.0]]])

print("Đầu vào:\n", example_input)

embed_dim = 4
num_heads = 2
num_key_value_groups = 1
gqa = GroupedQueryAttention(embed_dim, num_heads, num_key_value_groups)

output, attn_weights = gqa(example_input)

print("\nMa trận Attention Weights:\n", attn_weights)
print("\nKết quả đầu ra của Grouped-Query Attention:\n", output)


Đầu vào:
 tensor([[[1., 0., 1., 0.],
         [0., 2., 0., 2.],
         [1., 1., 1., 1.]]])

Ma trận Attention Weights:
 tensor([[[[0.2966, 0.3718, 0.3316],
          [0.4250, 0.1515, 0.4236],
          [0.3458, 0.2725, 0.3816]],

         [[0.3850, 0.2833, 0.3317],
          [0.3111, 0.3438, 0.3450],
          [0.3344, 0.3253, 0.3403]]]], grad_fn=<SoftmaxBackward0>)

Kết quả đầu ra của Grouped-Query Attention:
 tensor([[[-0.2003,  0.4006,  0.5708,  0.4208],
         [-0.2351,  0.4754,  0.7576,  0.3691],
         [-0.2086,  0.4372,  0.6599,  0.3894]]], grad_fn=<ViewBackward0>)





# FlashAttention v2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FlashAttentionV2(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(FlashAttentionV2, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert self.head_dim * num_heads == embed_dim, "embed_dim phải chia hết cho num_heads"

        self.W_QKV = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        batch_size, seq_length, _ = x.size()

        # Tính chung Q, K, V
        qkv = self.W_QKV(x)
        qkv = qkv.reshape(batch_size, seq_length, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        Q, K, V = qkv[0], qkv[1], qkv[2]

        # Attention scores với cơ chế FlashAttention v2 (chia nhỏ computation để tránh tràn bộ nhớ)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)

        # Attention output
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.embed_dim)

        output = self.out_proj(attn_output)

        return output, attn_weights


# ====== Ví dụ minh họa ======

# Đầu vào đơn giản: batch_size=1, seq_length=3, embed_dim=4
example_input = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                               [0.0, 2.0, 0.0, 2.0],
                               [1.0, 1.0, 1.0, 1.0]]])

print("Đầu vào:\n", example_input)

# Khởi tạo mô hình FlashAttention v2
embed_dim = 4
num_heads = 2
flash_attention = FlashAttentionV2(embed_dim, num_heads)

# Forward
output, attn_weights = flash_attention(example_input)

print("\nMa trận Attention Weights:\n", attn_weights)
print("\nKết quả đầu ra của FlashAttention v2:\n", output)


Đầu vào:
 tensor([[[1., 0., 1., 0.],
         [0., 2., 0., 2.],
         [1., 1., 1., 1.]]])

Ma trận Attention Weights:
 tensor([[[[0.3442, 0.3215, 0.3343],
          [0.2731, 0.3982, 0.3287],
          [0.3156, 0.3513, 0.3331]],

         [[0.4081, 0.2550, 0.3370],
          [0.4144, 0.2364, 0.3492],
          [0.4346, 0.2244, 0.3411]]]], grad_fn=<SoftmaxBackward0>)

Kết quả đầu ra của FlashAttention v2:
 tensor([[[-0.3489,  0.0410,  0.5035, -0.0705],
         [-0.3645,  0.0124,  0.4752, -0.0793],
         [-0.3637,  0.0359,  0.5060, -0.0876]]], grad_fn=<ViewBackward0>)


# RoPE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RoPEAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(RoPEAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert embed_dim % num_heads == 0, "embed_dim phải chia hết cho num_heads"

        self.W_QKV = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def apply_rope(self, x):
        seq_len = x.size(-2)
        dim = x.size(-1)
        position_ids = torch.arange(seq_len, dtype=torch.float, device=x.device)
        indices = torch.arange(0, dim, 2, dtype=torch.float, device=x.device)

        inv_freq = 1.0 / (10000 ** (indices / dim))
        sinusoid_inp = torch.einsum("i,j->ij", position_ids, inv_freq)

        sin = sinusoid_inp.sin()
        cos = sinusoid_inp.cos()

        x1, x2 = x[..., ::2], x[..., 1::2]
        x_rope = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

        return x_rope.flatten(-2)

    def forward(self, x):
        batch_size, seq_length, _ = x.size()

        qkv = self.W_QKV(x).view(batch_size, seq_length, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        Q, K, V = qkv[0], qkv[1], qkv[2]

        # Áp dụng RoPE
        Q = self.apply_rope(Q)
        K = self.apply_rope(K)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(attn_scores, dim=-1)

        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.embed_dim)

        output = self.out_proj(attn_output)

        return output, attn_weights


# ===== Ví dụ minh họa ======

# Đầu vào: batch_size=1, seq_length=3, embed_dim=4
example_input = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                               [0.0, 2.0, 0.0, 2.0],
                               [1.0, 1.0, 1.0, 1.0]]])

print("Đầu vào:\n", example_input)

# Khởi tạo mô hình RoPE Attention
embed_dim = 4
num_heads = 2
rope_attention = RoPEAttention(embed_dim, num_heads)

# Forward
output, attn_weights = rope_attention(example_input)

print("\nMa trận Attention Weights:\n", attn_weights)
print("\nKết quả đầu ra của RoPE Attention:\n", output)

Đầu vào:
 tensor([[[1., 0., 1., 0.],
         [0., 2., 0., 2.],
         [1., 1., 1., 1.]]])

Ma trận Attention Weights:
 tensor([[[[0.2568, 0.3256, 0.4176],
          [0.3024, 0.3206, 0.3770],
          [0.3013, 0.3178, 0.3809]],

         [[0.2323, 0.6173, 0.1504],
          [0.2405, 0.3997, 0.3598],
          [0.1550, 0.2018, 0.6432]]]], grad_fn=<SoftmaxBackward0>)

Kết quả đầu ra của RoPE Attention:
 tensor([[[ 0.3771, -0.7853,  0.1700,  0.8042],
         [ 0.3582, -0.8198,  0.2183,  0.8644],
         [ 0.3529, -0.8613,  0.2560,  0.9335]]], grad_fn=<ViewBackward0>)
